In [1]:
#==============================================
# Librerias
#==============================================
import pennylane as qml
import numpy as np
from scipy.integrate import quad

In [2]:
#========================================================
# Parámetros del Algoritmo QME
#========================================================

# Número de qubits de índice
n_qubits = 8  # N = 2^8 = 256 puntos de discretización

# Número total de puntos en la malla
N = 2**n_qubits

# Parámetro de la función integrando f(x) = x^p
p = 4

# Intervalo de integración
a = -1.0  # Límite inferior
b = 1.0   # Límite superior

# Crear dispositivo cuántico
dev = qml.device("default.qubit", wires=n_qubits+1)

# Malla de discretización
x_values = np.linspace(a, b, N)
f_values = x_values**p

# Encontrar el máximo valor del integrando para normalizar
f_max = np.max(np.abs(f_values))

# Integrando normalizado
f_norm = f_values / f_max


In [3]:
#========================================================
# Función Cuántica Principal (QME Circuit)
#========================================================

@qml.qnode(dev)
def qme_circuit():
    """
    Circuito cuántico que implementa la estimación cuántica 
    del valor medio (QME).
    """
    
    # Paso 1: Preparar superposición uniforme en los qubits de índice
    # Aplicar compuertas Hadamard a todos los qubits de índice
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    
    # Paso 2: Codificar el integrando en las amplitudes
    # Este paso es la operación unitaria U_psi que codifica f(x)
    # En una implementación simple, usamos rotaciones controladas
    
    # Para cada qubit de índice, aplicamos rotaciones RY controladas
    # que codifican el valor del integrando normalizado
    for i in range(n_qubits):
        # El ángulo de rotación se determina a partir de los valores normalizados
        # Mapeo: los qubits i=0 a i=n_qubits-1 representan diferentes bits
        
        # Crear rotaciones RY basadas en la información del integrando
        for k in range(2**i):
            idx = k * 2**(n_qubits - i)
            if idx < len(f_norm):
                # Ángulo basado en el valor normalizado del integrando
                theta = 2 * np.arcsin(f_norm[idx])
                # Aplicar rotación controlada por los qubits anteriores
                qml.RY(theta, wires=n_qubits)
    
    # Paso 3: Medir el observable Z en el qubit ancila
    # El qubit ancila es el qubit n_qubits (el último qubit)
    return qml.expval(qml.PauliZ(n_qubits))

In [4]:
#========================================================
# Ejecución del Circuito y Cálculo de la Integral
#========================================================

# Ejecutar el circuito QME
expectation_z = qme_circuit()

# Recuperar el valor de la integral
# I ≈ (b-a) * f_max * <Z> / N
integral_qme = (b - a) * f_max * expectation_z / N

# Calcular la integral exacta para comparación
integral_exact, _ = quad(lambda x: x**p, a, b)

# Método clásico de suma de Riemann (para comparación)
integral_riemann = np.sum(f_values) * (b - a) / N

In [5]:
#========================================================
# Visualizar el Circuito
#========================================================

print("=" * 60)
print("ALGORITMO QME - ESTIMACIÓN CUÁNTICA DEL VALOR MEDIO")
print("=" * 60)
print(f"\nParámetros del algoritmo:")
print(f"  - Número de qubits de índice: {n_qubits}")
print(f"  - Número de puntos de discretización: {N}")
print(f"  - Función integrando: f(x) = x^{p}")
print(f"  - Intervalo de integración: [{a}, {b}]")
print(f"  - Máximo valor del integrando: {f_max:.6f}")

print(f"\nDiagrama del circuito QME:")
print(qml.draw(qme_circuit)())

print(f"\nResultados:")
print(f"  - Valor esperado <Z>: {expectation_z:.6f}")
print(f"  - Integral QME: {integral_qme:.6f}")
print(f"  - Integral exacta (scipy): {integral_exact:.6f}")
print(f"  - Integral Riemann clásica: {integral_riemann:.6f}")

print(f"\nErrores relativos:")
error_qme = np.abs(integral_qme - integral_exact) / np.abs(integral_exact)
error_riemann = np.abs(integral_riemann - integral_exact) / np.abs(integral_exact)
print(f"  - Error QME: {error_qme*100:.4f}%")
print(f"  - Error Riemann: {error_riemann*100:.4f}%")

print("=" * 60)

ALGORITMO QME - ESTIMACIÓN CUÁNTICA DEL VALOR MEDIO

Parámetros del algoritmo:
  - Número de qubits de índice: 8
  - Número de puntos de discretización: 256
  - Función integrando: f(x) = x^4
  - Intervalo de integración: [-1.0, 1.0]
  - Máximo valor del integrando: 1.000000

Diagrama del circuito QME:
0: ──H───────────────────────────────────────────────────────────────────────────────────────
1: ──H───────────────────────────────────────────────────────────────────────────────────────
2: ──H───────────────────────────────────────────────────────────────────────────────────────
3: ──H───────────────────────────────────────────────────────────────────────────────────────
4: ──H───────────────────────────────────────────────────────────────────────────────────────
5: ──H───────────────────────────────────────────────────────────────────────────────────────
6: ──H───────────────────────────────────────────────────────────────────────────────────────
7: ──H────────────────────────────────